In [ ]:
%run "D:/pyspark_udemy_codespace/joins/data_prep.ipynb"
# run only to load data

In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
spark.sql("SELECT * FROM spark_db.facilities").show()
spark.sql("SELECT * FROM spark_db.members").show()
spark.sql("SELECT * FROM spark_db.bookings").show()

+-----+---------------+----------+---------+-------------+------------------+
|facid|       fac_name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+---------------+----------+---------+-------------+------------------+
|    0| Tennis Court 1|         5|       25|        10000|               200|
|    1| Tennis Court 2|         5|       25|         8000|               200|
|    2|Badminton Court|         0|     NULL|         4000|                50|
|    3|   Table Tennis|         0|        5|          320|                10|
|    4| Massage Room 1|        35|       80|         4000|              3000|
|    5| Massage Room 2|        35|       80|         4000|              3000|
|    6|   Squash Court|      NULL|     NULL|         5000|                80|
|    7|  Snooker Table|         0|        5|          450|                15|
|    8|     Pool Table|         0|        5|          400|                15|
+-----+---------------+----------+---------+-------------+------

In [4]:
"""
Prepare a facility bookings reporting dataset as the following.

member_id | first_name | last_name | facility_id | slots | start_time
---------------------------------------------------------------------------
The report must meet the following criteria.

    Facility bookings made by a person whose last name is Smith
    He has booked more than 5 slots in a single booking
    Report should be sorted by first name of the member in ascending order and number of slots in descending order
"""

spark.sql("""
    SELECT b.memid AS member_id, m.firstname, m.surname, b.facid, b.slots, b.starttime
    FROM spark_db.bookings b JOIN spark_db.members m ON b.memid = m.memid
    WHERE UPPER(m.surname) = 'SMITH' AND b.slots > 5
    ORDER BY m.firstname ASC, b.slots DESC
    
""").show()

+---------+---------+-------+-----+-----+-------------------+
|member_id|firstname|surname|facid|slots|          starttime|
+---------+---------+-------+-----+-----+-------------------+
|        1|   Darren|  Smith|    2|    9|2022-08-28 13:30:00|
|        1|   Darren|  Smith|    2|    6|2022-07-09 09:00:00|
|        1|   Darren|  Smith|    2|    6|2022-07-27 12:00:00|
|        1|   Darren|  Smith|    2|    6|2022-07-29 12:00:00|
|        1|   Darren|  Smith|    2|    6|2022-08-01 09:30:00|
|        1|   Darren|  Smith|    2|    6|2022-08-07 09:00:00|
|        1|   Darren|  Smith|    2|    6|2022-08-20 15:00:00|
|        1|   Darren|  Smith|    2|    6|2022-09-07 14:00:00|
|        1|   Darren|  Smith|    2|    6|2022-09-09 13:00:00|
|        1|   Darren|  Smith|    2|    6|2022-09-10 09:00:00|
|        1|   Darren|  Smith|    2|    6|2022-09-30 14:00:00|
|       14|     Jack|  Smith|    0|    6|2022-08-21 09:00:00|
|       14|     Jack|  Smith|    0|    6|2022-09-07 09:30:00|
|       

In [13]:
"""
Try answering with dataframe api

    Join Expression
    Join type
    Column name ambiguity
"""
from pyspark.sql.functions import col, expr, upper # type: ignore

members_df = spark.table("spark_db.members").alias("m")
bookings_df = spark.table("spark_db.bookings").alias("b")

# join_expr = expr("m.memid == b.memid")
join_expr = col("m.memid") == col("b.memid")

# params for join() function: df1.join(df2, join_expr, type_of_join) # members_df is driving df
reports_df = members_df.join(bookings_df, join_expr, "inner")\
                    .filter((upper(col("m.surname")) == 'SMITH') &
                            (col("b.slots") > 5))\
                    .select("m.memid", "m.firstname", "m.surname", "b.facid", "b.slots", "b.starttime")\
                    .orderBy(col("m.firstname").asc(), col("b.slots").desc())
reports_df.show()

+-----+---------+-------+-----+-----+-------------------+
|memid|firstname|surname|facid|slots|          starttime|
+-----+---------+-------+-----+-----+-------------------+
|    1|   Darren|  Smith|    2|    9|2022-08-28 13:30:00|
|    1|   Darren|  Smith|    2|    6|2022-07-09 09:00:00|
|    1|   Darren|  Smith|    2|    6|2022-07-27 12:00:00|
|    1|   Darren|  Smith|    2|    6|2022-07-29 12:00:00|
|    1|   Darren|  Smith|    2|    6|2022-08-01 09:30:00|
|    1|   Darren|  Smith|    2|    6|2022-08-07 09:00:00|
|    1|   Darren|  Smith|    2|    6|2022-08-20 15:00:00|
|    1|   Darren|  Smith|    2|    6|2022-09-07 14:00:00|
|    1|   Darren|  Smith|    2|    6|2022-09-09 13:00:00|
|    1|   Darren|  Smith|    2|    6|2022-09-10 09:00:00|
|    1|   Darren|  Smith|    2|    6|2022-09-30 14:00:00|
|   14|     Jack|  Smith|    0|    6|2022-08-21 09:00:00|
|   14|     Jack|  Smith|    0|    6|2022-09-07 09:30:00|
|   14|     Jack|  Smith|    0|    6|2022-09-24 15:30:00|
|    2|    Tra

In [14]:
"""
Show me a facility bookings report as the following.

member_id | first_name | last_name | facility_name | slots | booking_amount | start_time
--------------------------------------------------------------------------------------------
The report must meet the following criteria.

    Facility bookings made by a person whose last name is Smith
    He has booked more than 5 slots in a single booking
    Report should be sorted by first name of the member in ascending order and booking amount in descending order
"""

facilities_df = spark.table("spark_db.facilities").alias("f")

booking_members_join = col("b.memid") == col("m.memid")
booking_facilities_join = col("b.facid") == col("f.facid")

reports2_df = bookings_df.join(members_df, booking_members_join, "inner")\
                        .join(facilities_df, booking_facilities_join, "inner")\
                        .filter(
                                (upper(col("m.surname")) == 'SMITH') &
                                (col("b.slots") > 5)
                        )\
                        .select(
                                col("b.memid").alias("member_id"), 
                                col("m.firstname"), 
                                col("m.surname").alias("lastname"), 
                                col("f.fac_name").alias("facility_name"), 
                                col("b.slots"), 
                                (col("b.slots") * col("f.membercost")).alias("booking_amount"),
                                (col("b.starttime"))
                        )\
                        .orderBy(col("m.firstname").asc(), col("booking_amount").desc())
reports2_df.show()

+---------+---------+--------+---------------+-----+--------------+-------------------+
|member_id|firstname|lastname|  facility_name|slots|booking_amount|          starttime|
+---------+---------+--------+---------------+-----+--------------+-------------------+
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-07-09 09:00:00|
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-07-27 12:00:00|
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-07-29 12:00:00|
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-08-01 09:30:00|
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-08-07 09:00:00|
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-08-20 15:00:00|
|        1|   Darren|   Smith|Badminton Court|    9|             0|2022-08-28 13:30:00|
|        1|   Darren|   Smith|Badminton Court|    6|             0|2022-09-07 14:00:00|
|        1|   Darren|   Smith|Ba